# Aula 6 — Quantum ESPRESSO on Google Colab

Hands-on: silicon SCF, convergence tests (`ecutwfc`, k mesh) and the lattice parameter.
Run the cells in order. Every number quoted below was checked with QE 6.7 (the version `apt` installs on Colab).

## 1. Install Quantum ESPRESSO
About 1 min. Re-run this cell every time the Colab session restarts (your files on Drive stay). The last line should print `/usr/bin/pw.x`.

**Why the extra lines?** Colab now runs Ubuntu 24.04, whose `quantum-espresso` 6.7 package aborts at startup with `*** buffer overflow detected ***`. It is a false alarm of a safety check (`_FORTIFY_SOURCE`): QE writes the MD5 checksum of the pseudopotential with `snprintf`, telling it the buffer is larger than it really is, and the check aborts even though nothing is written out of bounds. The fix (`src/qe_fix.c` in this repository) compiles a tiny replacement for that check, which truncates instead of aborting, and loads it in every program (`/etc/ld.so.preload`), so `pw.x` works both here and in the Terminal. Results are unchanged: −15.85039422 Ry for the silicon SCF.

In [ ]:
# Quantum ESPRESSO from Ubuntu's package manager (about 1 min).
# In Colab, a line that starts with "!" is sent to the Linux shell instead of Python.

# Refresh the list of available packages (-qq: print almost nothing).
!apt-get update -qq

# Install the quantum-espresso package: pw.x, ph.x, dos.x, bands.x, ...
# -y answers "yes" to every question; "> /dev/null" hides the long installation log
# (remove it if the installation fails and you want to see the error).
!apt-get install -y -qq quantum-espresso > /dev/null

# Fix for Ubuntu 24.04 (see the text above): without it pw.x stops with
# "*** buffer overflow detected ***". Download the small C file that fixes it ...
!curl -sL https://raw.githubusercontent.com/flavianojs/dft-cbpf/main/src/qe_fix.c -o /tmp/qe_fix.c
# ... compile it into a shared library ...
!gcc -O2 -shared -fPIC -o /usr/local/lib/qe_fix.so /tmp/qe_fix.c
# ... and ask Linux to load it in every program, including pw.x run from the Terminal.
!echo /usr/local/lib/qe_fix.so > /etc/ld.so.preload

# Check that the main program, pw.x (PWscf), is now on the PATH.
# Expected output: /usr/bin/pw.x
!which pw.x

## 2. Get the course material into your Google Drive
The Colab machine is wiped when the session ends; Google Drive is not. The clone is skipped if the folder already exists, so this cell is safe to re-run in every session.

In [ ]:
# Give this notebook access to your Google Drive.
# Colab asks for authorization the first time; your Drive then appears at /content/drive/MyDrive.
from google.colab import drive
drive.mount("/content/drive")

# "%cd" (with %, not !) changes the working folder of the whole notebook.
# ("!cd" would only change it for that single line.)
%cd /content/drive/MyDrive

# Address of the course repository on GitHub.
REPO = "https://github.com/flavianojs/dft-cbpf.git"

# Clone the repository only if the folder does not exist yet:
#   [ -d dft-cbpf ]  is true if the folder already exists;
#   "||"            runs the git command only when that test fails.
# --depth 1 downloads only the latest version; {REPO} is replaced by the Python variable above.
![ -d dft-cbpf ] || git clone -q --depth 1 {REPO}

# Work inside dft-cbpf/calculations from now on: all paths below are relative to it.
%cd dft-cbpf/calculations

## 3. First run by hand (Terminal)
Open the **Terminal** (button at the bottom left of Colab) and type:

```bash
cd /content/drive/MyDrive/dft-cbpf/calculations
mkdir -p aula6
cp ../src/silicon/pw.scf.silicon.in aula6/
```

In the file panel (folder icon on the left) go to drive › MyDrive › dft-cbpf › calculations › aula6, open `pw.scf.silicon.in` and change one line, then save (Ctrl+S):

```
pseudo_dir = '../pseudos/'        ->   pseudo_dir = '../../src/pseudos'
```

Back in the Terminal:

```bash
cd aula6
pw.x < pw.scf.silicon.in > pw.scf.silicon.out
grep "total energy *=" pw.scf.silicon.out
```

Expected: four iterations, the last one `!    total energy = -15.85039422 Ry` (same as the reference output in `calculations/reference/`).

## 4. Automate: four helper functions
What we did by hand (copy, edit one line, run, grep) becomes four functions, so we can run dozens of calculations. Each run goes to `calculations/aula6/`.

In [ ]:
import re                  # regular expressions: find and replace patterns in text
import subprocess          # run external programs (here: pw.x) from Python
from pathlib import Path   # handle file and folder paths


def set_var(text, name, value):
    """Return a copy of the input text with the namelist line  name = ...  set to value.

    This is what we did by hand in the editor: change one line of the input.
    Example: set_var(text, "ecutwfc", 40) turns "ecutwfc = 30" into "ecutwfc = 40".
    """
    # Pattern for the start of the line: optional spaces, the name, optional spaces, "=".
    # re.escape protects characters such as the parentheses in "celldm(1)".
    pat = rf"^(\s*{re.escape(name)}\s*=\s*).*$"
    # Keep that start (group 1, written \g<1>) and put the new value after it.
    # re.M: ^ and $ match at every line; re.I: ignore upper/lower case (Fortran style).
    # count=1: change only the first match. n is the number of replacements made.
    text, n = re.subn(pat, rf"\g<1>{value}", text, count=1, flags=re.M | re.I)
    # Stop with a clear message if the variable is not in the input (e.g. a typo).
    assert n == 1, f"{name} not found"
    return text


def set_kpts(text, mesh):
    """Return a copy of the input with the line after K_POINTS replaced by mesh.

    Example: set_kpts(text, "8 8 8 1 1 1") -> 8x8x8 Monkhorst-Pack mesh, shifted.
    """
    # Group 1 is the K_POINTS line itself (kept); ".*" after it is the next line (replaced).
    return re.sub(r"(K_POINTS.*\n).*", rf"\g<1>  {mesh}", text, count=1)


def run(text, name, workdir="aula6"):
    """Write workdir/name.in, run pw.x on it, and return the output as text.

    This is what we did by hand in the Terminal:  pw.x < name.in > name.out
    """
    w = Path(workdir)
    w.mkdir(exist_ok=True)                 # create the folder if needed (no error if it exists)
    (w / f"{name}.in").write_text(text)    # save the input file
    # Run pw.x inside the folder (cwd=w), so relative paths in the input
    # (pseudo_dir, outdir) are relative to it; everything pw.x prints goes to name.out.
    with open(w / f"{name}.out", "w") as out:
        subprocess.run(["pw.x", "-in", f"{name}.in"], cwd=w, stdout=out)
    return (w / f"{name}.out").read_text() # read the output back


def etot(out):
    """Return the converged total energy (in Ry) from a pw.x output.

    This is our grep: the converged energy is on the line that starts with "!".
    """
    # \s+ = one or more spaces; (\S+) captures the number after the "=" sign.
    # findall returns every match; [-1] takes the last one (useful for relaxations).
    return float(re.findall(r"!\s+total energy\s+=\s+(\S+)", out)[-1])

## 5. The same SCF from Python
Expected: same four iterations and `-15.85039422 Ry` as the run by hand. A new name (`si_scf`) keeps the hand-made output.

In [ ]:
# Read the original input from src/ as one long string (the file itself is never modified).
base = Path("../src/silicon/pw.scf.silicon.in").read_text()

# Same edit we did by hand: pw.x will run inside calculations/aula6/,
# and from there the pseudopotentials are in ../../src/pseudos.
# Note the inner quotes: in the input file the value must be written as '../../src/pseudos'.
inp = set_var(base, "pseudo_dir", "'../../src/pseudos'")

# Run it under a new name (si_scf) so the output we made by hand is not overwritten.
out = run(inp, "si_scf")

# Line 2 of the output says which version ran (expected: v.6.7MaX).
print(out.splitlines()[1].strip())

# One line per SCF iteration; the converged one starts with "!".
!grep "total energy *=" aula6/si_scf.out

# From now on, "inp" is the starting point of every test: we change one parameter at a time.

## 6. Convergence with `ecutwfc` (k mesh fixed at 6×6×6, centered)
Expected: 30 Ry is 8.4 meV/atom above 60 Ry; 1 meV/atom is reached between 40 and 48 Ry.

In [ ]:
import numpy as np                 # arrays and numerics
import matplotlib.pyplot as plt    # plots

RY_MEV = 13605.693                 # 1 Ry = 13605.693 meV

# Cutoffs to test, in Ry. The last one (60 Ry) plays the role of the converged value.
ecuts = [12, 16, 20, 24, 28, 32, 36, 40, 48, 60]

E = []                             # total energies, in Ry
for ec in ecuts:
    inp_ec = set_var(inp, "ecutwfc", ec)    # same input, only ecutwfc changed
    out = run(inp_ec, f"ecut{ec}")          # files aula6/ecut12.in, aula6/ecut12.out, ...
    E.append(etot(out))
    print(f"ecutwfc = {ec:3d} Ry   E = {E[-1]:.8f} Ry")
E = np.array(E)

# Error with respect to 60 Ry, per atom (2 atoms in the cell), in meV.
dE = (E - E[-1]) / 2 * RY_MEV

# Plot 1, linear scale: the energy drops fast and then seems flat.
# Can you tell from it where the error is below 1 meV/atom?
plt.plot(ecuts, dE, "o-")
plt.axhline(1, ls="--")            # the 1 meV/atom criterion
plt.xlabel("ecutwfc (Ry)")
plt.ylabel("E - E(60 Ry) (meV/atom)")
plt.title("Linear scale")
plt.show()

# Plot 2, same data with a log scale on y: now every decade of the error is visible,
# and the 1 meV/atom line can be read off. The 60 Ry point (dE = 0) is left out (log of 0).
plt.semilogy(ecuts[:-1], dE[:-1], "o-")
plt.axhline(1, ls="--")            # the 1 meV/atom criterion
plt.xlabel("ecutwfc (Ry)")
plt.ylabel("E - E(60 Ry) (meV/atom)")
plt.title("Log scale on y (semilog)")
plt.show()

## 7. Convergence with the k mesh (`ecutwfc` = 30 Ry)
Expected: centered 6×6×6 → 12 meV/atom; shifted 6×6×6 → 0.04 meV/atom. The 16×16×16 reference takes ~20 s.

In [ ]:
import time                        # to show how long each calculation takes

# Meshes n x n x n to test; ecutwfc stays at 30 Ry (the value in inp).
ks = [2, 4, 6, 8, 10, 12]

def nks(out):
    """Number of irreducible k points that pw.x used (printed in the output)."""
    return int(re.findall(r"number of k points=\s*(\d+)", out)[0])

# Reference: a dense 16x16x16 shifted mesh (408 irreducible k points, ~20 s).
# flush=True makes Colab show each line right away, while the loop is still running.
print("Reference 16x16x16 shifted: running ...", flush=True)
t0 = time.time()
out = run(set_kpts(inp, "16 16 16 1 1 1"), "k16s1")
E_ref = etot(out)
print(f"Reference: E = {E_ref:.8f} Ry  ({nks(out)} k points, {time.time() - t0:.1f} s)", flush=True)

# The last three numbers of the K_POINTS line are the shift:
#   0 0 0 -> centered mesh (contains Gamma);  1 1 1 -> shifted by half a step.
errors = {}                        # errors[0]: centered meshes, errors[1]: shifted meshes
for s in (0, 1):
    print(f"\n{'Centered' if s == 0 else 'Shifted'} meshes (shift {s} {s} {s}):", flush=True)
    E = []
    for n in ks:
        mesh = f"{n} {n} {n} {s} {s} {s}"           # e.g. "6 6 6 0 0 0"
        t0 = time.time()
        out = run(set_kpts(inp, mesh), f"k{n}s{s}")  # files aula6/k6s0.in, aula6/k6s0.out, ...
        E.append(etot(out))
        err = abs(E[-1] - E_ref) / 2 * RY_MEV        # meV/atom
        print(f"  {f'{n}x{n}x{n}':>8}: {nks(out):3d} k points   E = {E[-1]:.8f} Ry   "
              f"|E - E_ref| = {err:9.4f} meV/atom   ({time.time() - t0:.1f} s)", flush=True)
    # Error per atom in meV, kept for the plots below.
    # abs(): near convergence the difference can change sign (k sampling is not variational).
    errors[s] = abs(np.array(E) - E_ref) / 2 * RY_MEV

# Plot 1, linear scale: both meshes seem converged beyond n = 6.
for s in (0, 1):
    plt.plot(ks, errors[s], "o-", label=f"shift {s}")
plt.axhline(1, ls="--")            # the 1 meV/atom criterion
plt.legend()
plt.xlabel("n (n x n x n mesh)")
plt.ylabel("|E - E_ref| (meV/atom)")
plt.title("Linear scale")
plt.show()

# Plot 2, same data with a log scale on y: the shifted mesh is orders of magnitude better.
for s in (0, 1):
    plt.semilogy(ks, errors[s], "o-", label=f"shift {s}")
plt.axhline(1, ls="--")            # the 1 meV/atom criterion
plt.legend()
plt.xlabel("n (n x n x n mesh)")
plt.ylabel("|E - E_ref| (meV/atom)")
plt.title("Log scale on y (semilog)")
plt.show()

## 8. Lattice parameter (40 Ry, shifted 8×8×8)
Expected: `a0 = 10.208 bohr = 5.402 Å` (experiment: 5.431 Å).

In [ ]:
# Converged parameters, chosen from the two tests above: 40 Ry and an 8x8x8 shifted mesh.
inp_a = set_kpts(set_var(inp, "ecutwfc", 40), "8 8 8 1 1 1")

# Lattice parameters a = celldm(1), in bohr: 9.9, 10.0, ..., 10.6 (round(2) avoids 10.000000001).
alats = np.arange(9.9, 10.61, 0.1).round(2)

E = []
for a in alats:
    out = run(set_var(inp_a, "celldm(1)", a), f"a{a:.2f}")   # files aula6/a9.90.in, ...
    E.append(etot(out))
    print(f"a = {a:.2f} bohr   E = {E[-1]:.8f} Ry")
E = np.array(E)

# Fit E(a) with a cubic polynomial; the minimum is where the derivative vanishes.
c = np.polyfit(alats, E, 3)          # coefficients of the cubic
r = np.roots(np.polyder(c)).real     # the two zeros of dE/da (a quadratic)
a0 = r[np.argmin(abs(r - 10.2))]     # keep the zero near 10.2 bohr: the minimum
print(f"a0 = {a0:.3f} bohr = {a0 * 0.529177:.3f} Å")   # 1 bohr = 0.529177 Å

# Plot energies relative to the lowest point, per atom, in meV, with the fit.
# Colors: orange = our LDA calculation (points, fit, minimum); blue = experiment.
plt.plot(alats, (E - E.min()) / 2 * RY_MEV, "o", color="C1", label="pw.x")
x = np.linspace(9.85, 10.65, 200)
plt.plot(x, (np.polyval(c, x) - E.min()) / 2 * RY_MEV, color="C1", label="cubic fit")
plt.axvline(a0, ls=":", color="C1", label="LDA minimum")
plt.axvline(10.263, ls="--", color="C0", label="experiment (5.431 Å)")
plt.xlabel("celldm(1) (bohr)")
plt.ylabel("E - E_min (meV/atom)")
plt.legend(loc="lower right")   # the empty corner of the plot
plt.show()

## Extras
### vc-relax
`relax` and `vc-relax` need the `&IONS` and `&CELL` namelists, even if empty. Expected: `CELL_PARAMETERS` with 0.497332426 instead of 0.5 → a = 10.26 × 0.497332/0.5 = 10.205 bohr.

In [ ]:
# vc-relax: pw.x moves the atoms AND changes the cell until forces and stress vanish.
vc = set_var(inp_a, "calculation", "'vc-relax'")

# relax and vc-relax need the &IONS and &CELL namelists (empty = default algorithms),
# placed after &ELECTRONS. We insert them just before the ATOMIC_SPECIES card.
vc = vc.replace("ATOMIC_SPECIES", "&IONS\n/\n&CELL\n/\nATOMIC_SPECIES")

out = run(vc, "vcrelax")   # ~15 s: a few SCF cycles, one per BFGS step

# The relaxed cell is printed between "Begin final coordinates" and "End final coordinates".
# The new a = 10.26 x (first number of CELL_PARAMETERS, in absolute value) / 0.5
!grep -A6 "Begin final coordinates" aula6/vcrelax.out

### Number of plane waves at Γ: 30 vs 60 Ry
Does it grow like 2^(3/2) ≈ 2.8?

In [ ]:
# Number of plane waves at Gamma (k = 0 0 0), printed because the input has verbosity = 'high'.
# -m1: show only the first match in each file. Compare 30 Ry (hand-made run) with 60 Ry.
!grep -m1 "PWs)" aula6/pw.scf.silicon.out aula6/ecut60.out

### Next session
Re-run sections 1 and 2 (QE is reinstalled, the clone is skipped), then the cells of sections 4 and 5 to define the functions and `inp` again. Everything in `aula6/` is still in your Drive.